# Lab 3 - Web Application Security

Interactive notebook. For each part: **run** the demo, **implement** the function in the *Your code* cell, then **run the check cell** below it (it should print `PASS`).

Full theory and the research tasks are in `README.md`. Everything uses the Python standard library only.

In [ ]:
# Helper used by the check cells.
def check(name, cond):
    print(f"{'PASS' if cond else 'FAIL'}  {name}")
    return bool(cond)

def expect_error(fn, *a, exc=Exception):
    try: fn(*a)
    except exc: return True
    except Exception: return False
    return False

## Part A - Output encoding & XSS  ·  A05:2025 Injection

Untrusted input written into a page without encoding is executed by the browser.
Fix it with **context-aware output encoding**, and never allow a `javascript:` URL in an `href`.

In [ ]:
from demo_xss import render_profile_vulnerable, render_profile_safe, link_safe, PAYLOAD
print('vulnerable:', render_profile_vulnerable(PAYLOAD))
print('safe:      ', render_profile_safe(PAYLOAD))
print('href safe: ', link_safe('javascript:alert(document.cookie)'))

**A.3 Your code** - implement HTML text encoding and an href URL check.

In [ ]:
import html
from urllib.parse import urlparse

def encode_html_text(s):
    # TODO: HTML-encode for the text context (no raw <script>)
    raise NotImplementedError

def is_safe_url(url):
    # TODO: allow relative or http/https; reject javascript:/data:/vbscript:
    raise NotImplementedError

In [ ]:
# Check - Part A
try:
    out = encode_html_text("<script>alert('XSS')</script>")
    check('encode neutralizes script', '<script>' not in out and '&lt;script&gt;' in out)
    check('safe urls allowed', all(is_safe_url(u) for u in ['/x','https://a.b/c']))
    check('dangerous urls blocked', not any(is_safe_url(u) for u in ['javascript:x',' JavaScript:x','data:x']))
except NotImplementedError:
    print('TODO: implement the two functions above')

## Part B - Cookies & sessions  ·  A07:2025 Authentication Failures

Session IDs are credentials: strong, unique, correctly flagged, and **rotated on login** to defeat fixation.

In [ ]:
from demo_sessions import weak_session_id, strong_session_id, weak_cookie, strong_cookie
print('weak id:   ', weak_session_id())
print('strong id: ', strong_session_id())
print('weak cookie:  ', weak_cookie('SID','abc'))
print('strong cookie:', strong_cookie('SID','abc'))

**B.3 Your code** - a strong ID, a hardened cookie, a cookie audit, and session rotation.

In [ ]:
import secrets

def generate_session_id():
    # TODO: secrets.token_urlsafe(32) or similar
    raise NotImplementedError

def build_session_cookie(name, value):
    # TODO: include Secure; HttpOnly; SameSite=Lax; Path=/
    raise NotImplementedError

def audit_cookie(header):
    # TODO: return the set of missing-flag problems
    raise NotImplementedError

def rotate_session(store, old_sid):
    # TODO: new strong id, move data, delete old, return new id
    raise NotImplementedError

In [ ]:
# Check - Part B
try:
    ids = {generate_session_id() for _ in range(300)}
    check('ids strong & unique', len(ids)==300 and all(len(i)>=32 for i in ids))
    ck = build_session_cookie('SID','abc')
    check('cookie hardened', all(f in ck for f in ['Secure','HttpOnly','SameSite','Path=/']))
    check('audit flags weak cookie', audit_cookie('SID=x; Path=/')=={'missing Secure','missing HttpOnly','missing SameSite'})
    store={'old':{'user':'alice'}}; new=rotate_session(store,'old')
    check('rotation defeats fixation', new!='old' and 'old' not in store and store[new]['user']=='alice')
except NotImplementedError:
    print('TODO: implement the four functions above')

## Part C - CSRF & open redirects  ·  A01:2025 Broken Access Control

A CSRF token the attacker cannot predict proves a request came from your own page.
Compare it in **constant time**. Validate redirect targets against an **allow-list**.

In [ ]:
from demo_requests import make_token, check_token, redirect_safe
sid='session-abc'; tok=make_token(sid)
print('valid   :', check_token(sid, tok))
print('tampered:', check_token(sid, tok[:-1]+'0'))
allowed={'example.com'}
for t in ['/home','https://example.com/x','//evil.example','https://example.com@evil.example']:
    print(f'  {t:<38} -> {redirect_safe(t, allowed)}')

**C.3 Your code** - CSRF generate/verify and a safe-redirect check.

In [ ]:
import hmac, hashlib
from urllib.parse import urlparse

def generate_csrf_token(session_id, secret):
    # TODO: HMAC-SHA256(secret, session_id).hexdigest()
    raise NotImplementedError

def verify_csrf_token(session_id, secret, token):
    # TODO: recompute and hmac.compare_digest(...)
    raise NotImplementedError

def safe_redirect_target(target, allowed_hosts):
    # TODO: allow relative or http(s) to an allowed host; else None
    raise NotImplementedError

In [ ]:
# Check - Part C
S=b'secret'
try:
    sid='s1'; tok=generate_csrf_token(sid,S)
    check('valid token verifies', verify_csrf_token(sid,S,tok) is True)
    check('tampered token fails', verify_csrf_token(sid,S,tok[:-1]+('0' if tok[-1]!='0' else '1')) is False)
    a={'example.com'}
    check('relative allowed', safe_redirect_target('/home',a)=='/home')
    bad=['https://evil.example','//evil.example','https://example.com@evil.example','javascript:alert(1)']
    check('evil targets blocked', all(safe_redirect_target(b,a) is None for b in bad))
except NotImplementedError:
    print('TODO: implement the three functions above')

## Part D - XXE & insecure deserialization  ·  A02 / A08:2025

Refuse XML documents that declare a **DTD** (blocks external entities and the billion-laughs bomb before anything expands).
For data exchange, use **JSON + an integrity signature**, not native deserialization.

*The billion-laughs string below is never expanded - the safe parser rejects it on sight.*

In [ ]:
from demo_parsers import parse_safe, sign, load, NORMAL_XML, EXTERNAL_ENTITY, BILLION_LAUGHS
print('normal root:', parse_safe(NORMAL_XML).tag)
for name,doc in [('external entity',EXTERNAL_ENTITY),('billion laughs',BILLION_LAUGHS)]:
    try: parse_safe(doc)
    except ValueError as e: print(f'rejected ({name}):', e)
print('signed round-trip:', load(sign({'user':'alice'})))

**D.3 Your code** - a DTD-refusing XML parser and a signed JSON payload format.

In [ ]:
import base64, json, hmac, hashlib
import xml.etree.ElementTree as ET

def parse_xml_safely(xml_str):
    # TODO: raise ValueError on '<!DOCTYPE', else ET.fromstring(...)
    raise NotImplementedError

def sign_payload(obj, key):
    # TODO: base64url(json).'.'.hmac_sha256
    raise NotImplementedError

def load_signed(token, key):
    # TODO: verify hmac (constant time), then decode; ValueError on tamper
    raise NotImplementedError

In [ ]:
# Check - Part D
K=b'k'
try:
    check('normal xml parses', parse_xml_safely('<u><n>Alice</n></u>').find('n').text=='Alice')
    check('DTD rejected', expect_error(parse_xml_safely, '<!DOCTYPE x [ <!ENTITY e \"e\"> ]><x>&e;</x>', exc=ValueError))
    t=sign_payload({'user':'alice'},K)
    check('signed round-trip', load_signed(t,K)=={'user':'alice'})
    check('tamper detected', expect_error(load_signed, t[:-1]+('0' if t[-1]!='0' else '1'), K, exc=ValueError))
except NotImplementedError:
    print('TODO: implement the three functions above')

## Done?

When every check prints `PASS`, copy your functions into `student_solutions.py` and run the full suite:

```bash
python -m pytest test_solutions.py -v
```